In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor,StackingRegressor


import warnings
warnings.filterwarnings('ignore')


In [41]:
df = pd.read_csv("bangladesh_student_performance.csv")
display(df.head())

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,29/04/2018,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,29/04/2018,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,29/04/2018,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,29/04/2018,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [42]:

from ydata_profiling import ProfileReport
profile = ProfileReport(df,title='Bangladesh Student Performance Prediction', explorative=True)
profile.to_file("bangladesh_student_performance_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 340.39it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [43]:
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'Pstatus', 'M_Edu',
       'F_Edu', 'M_Job', 'F_Job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [44]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'pstatus', 'm_edu',
       'f_edu', 'm_job', 'f_job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [45]:
df.drop(columns=['date'],inplace=True)

In [46]:

#^ correlation for numerical values
corr_target = df.select_dtypes(include=np.number).corr()['hsc_result'].sort_values(ascending=False)
corr_target


hsc_result      1.000000
ssc_result      0.950178
m_edu           0.063776
f_edu           0.054811
tuition_fee     0.038068
age            -0.009857
time_friends   -0.156356
Name: hsc_result, dtype: float64

In [47]:

#^ Separate X and y
X = df.drop(columns=['hsc_result'],axis=1) # means axis=1 for columns
y = df['hsc_result']



In [48]:

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler())
])

In [49]:
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [50]:
# Define numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# combine them
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_features),
        ("cat", cat_transformer, categorical_features),
    ]
)

In [51]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1614, 14), (404, 14), (1614,), (404,))

In [52]:

#& Base Learner
reg_lr = LinearRegression()
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)
reg_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)


In [53]:

#& Voting regressor
voting_reg = VotingRegressor(estimators=[
    ('lr', reg_lr),
    ('rf', reg_rf),
    ('gb', reg_gb)
])

In [54]:
# ^ Stacking
stacking_reg = StackingRegressor(
    estimators=[
        ("lr", reg_lr),
        ("rf", reg_rf)
    ],
    final_estimator=Ridge() #* the meta learner
)

### Model Training  

In [55]:

#^ dict of all model
model_to_train       =          {
    'Linear Regression': reg_lr,
    'Random Forest': reg_rf,
    'Gradient Boasting': reg_gb,
    'Voting Ensemble': voting_reg,
    'Stacking Ensemble': stacking_reg
}

In [56]:

#& Training and Evaluation
results = []

for name, model in model_to_train.items():
    # create full pipeline with preprocessor and model
    pipe = Pipeline(
        [
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )

    # Train
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_test - y_pred))

    results.append({
        'model': name,
        'mse': mse,
        'r2': r2,
        'rmse': rmse,
        'mae': mae
    })

results_df = pd.DataFrame(results).sort_values(by='r2', ascending=False)
results_df

,model,mse,r2,rmse,mae
2,Gradient Boasting,0.015155,0.959565,0.123107,0.098902
3,Voting Ensemble,0.015919,0.957528,0.126169,0.100838
4,Stacking Ensemble,0.016980,0.954697,0.130307,0.103853
1,Random Forest,0.018647,0.950248,0.136556,0.108201
0,Linear Regression,0.020269,0.945920,0.142371,0.111376


In [57]:

#^ Visualize the results
best_model = results_df.iloc[0]['model']
best_model_obj = model_to_train[best_model]
print(f"Best model based on R2 score: {best_model}")

final_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', best_model_obj)
    ]
)

final_pipe.fit(X_train, y_train)
y_final_pred = final_pipe.predict(X_test)




Best model based on R2 score: Gradient Boasting


In [58]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_final_pred)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # Line for perfect predictions
plt.xlabel('Actual HSC Result')
plt.ylabel('Predicted HSC Result')
plt.title(f'Actual vs Predicted HSC Result ({best_model})')
plt.show()



In [59]:

#^ Cross Validation
from sklearn.model_selection import cross_val_score
rf_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', reg_rf)
    ]
)

# 5 fold cross validation
cv_scores = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"Cross-validated RMSE for Random Forest: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")


Cross-validated RMSE for Random Forest: 0.1422 ± 0.0083
